# CAREPILOT — Priority Prediction Benchmark

*Chapter 2 — État de l'art et benchmarking IA · CAREPILOT operational AI*

## 1. Objective

Predict operational ticket **priority level** (`P1`–`P4`) before manual triage.

## 2. Problem Type

- **Multiclass classification** (4 priority levels)
- Primary metric: **Macro F1**

## 3. Dataset Description


- **Source:** `ml_training.historical_tickets` (14,000 tickets)
- **Class distribution:** P3 41.8%, P2 27.0%, P4 16.0%, P1 15.3%
- **Split:** 9800 / 2100 / 2100 (stratified)

## 4. Preprocessing

TF-IDF on ticket text, categorical encoding, numeric scaling. Priority excluded from features to avoid leakage.

## 5. Models / Methods Compared

Logistic Regression · Random Forest · XGBoost

This benchmark compares multiple approaches in order to select the most suitable method for integration into CAREPILOT.

## 6. Evaluation Metrics

Accuracy · Macro/Weighted Precision · Macro/Weighted Recall · Macro F1 · Weighted F1

## 7. Training or Retrieval Pipeline

Multiclass sklearn pipelines evaluated with Macro F1 on held-out test tickets.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR
for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (parent / "ml" / "artifacts").exists():
        ROOT = parent
        break

SHARED = ROOT / "ml" / "notebooks" / "_shared"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from benchmark_notebook_helpers import display_plot, find_project_root, plot_results_table

ROOT = find_project_root()
ARTIFACTS = ROOT / "ml" / "artifacts"
PLOTS = ROOT / "ml" / "plots"
REPORTS = ROOT / "ml" / "reports"
MODELS = ARTIFACTS / "models"

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

RUN_BENCHMARK = False  # Set True to re-train / re-run the full benchmark pipeline



In [ ]:
BENCHMARK_SCRIPT = ROOT / "ml/benchmarks/priority_prediction/benchmark.py"
if RUN_BENCHMARK:
    cmd = [sys.executable, str(BENCHMARK_SCRIPT), "--project-root", str(ROOT), *[]]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Skipping benchmark run (RUN_BENCHMARK=False). Loading saved artifacts.")
    print("Re-run with RUN_BENCHMARK=True or execute:")
    print(f"  python {BENCHMARK_SCRIPT} --project-root {ROOT}")



## 8. Benchmark Execution — Load Results

In [ ]:
metrics_path = ARTIFACTS / "03_priority_prediction_benchmark_metrics.json"
if not metrics_path.exists():
    raise FileNotFoundError(f"Missing metrics artifact: {metrics_path}")
payload = json.loads(metrics_path.read_text(encoding="utf-8"))
results_df = pd.DataFrame(payload.get('results', []))
display(results_df)


## 9. Benchmark Results

Final comparison table (report-ready PNG + DataFrame):

In [ ]:
table_png = PLOTS / "03_priority_benchmark_results_table.png"
display_plot(table_png)
display_plot(PLOTS / "03_priority_distribution.png")
display_plot(PLOTS / "03_priority_model_comparison.png")
display_plot(PLOTS / "03_priority_macro_f1.png")



## 10. Visualizations

Key plots are loaded from `ml/plots/` (generated by the benchmark pipeline). Re-run with `RUN_BENCHMARK=True` to regenerate.

## 11. Interpretation

XGBoost achieves Macro F1 ~0.83, clearly outperforming linear and forest baselines. P3 dominance in the dataset makes Macro F1 essential for fair evaluation across P1–P4.

## 12. Selected Model / Method

**Selected for CAREPILOT:**

**XGBoost** — best Macro F1. Artifact: `ml/artifacts/models/xgboost_priority.joblib`.

## 13. Limitations

- Priority labels may reflect subjective triage
- P1/P4 minority classes harder to classify
- Should be combined with SLA and escalation signals in operations

## 14. Generated Artifacts

- `ml/artifacts/03_priority_prediction_benchmark_metrics.json`
- `ml/plots/03_priority_benchmark_results_table.png`
- `ml/reports/03_priority_prediction_benchmark_summary.md`